In [12]:
!mkdir ../data
!wget https://data.ncl.ac.uk/ndownloader/articles/24574729/versions/1 -O ../data/Language.zip
!mkdir ../data/Language
!unzip ../data/Language.zip -d ../data/Language/
!rm ../data/Language.zip

--2025-05-21 21:14:23--  https://data.ncl.ac.uk/ndownloader/articles/24574729/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 54.194.125.230, 54.171.254.99, 52.50.254.254
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|54.194.125.230|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 161843534 (154M) [application/zip]
Saving to: ‘../data/Language.zip’

../data/Language.zi  20%[===>                ]  31.82M  1.34MB/s    eta 66s    ^C
Archive:  ../data/Language.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ../data/Language.zip or
        ../data/Language.zip.zip, and cannot find ../data/Language.zip.ZIP, period.


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from senmodel.model.utils import *
from senmodel.metrics.nonlinearity_metrics import *
from senmodel.metrics.edge_finder import *
from senmodel.metrics.train_metrics import *
from senmodel.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=24 * 24, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [3]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.2},
    "pruning_thresholds": {"fc0": 0.1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 24,
    "plateau_window_size": 5,
    "learning_rate": 8e-4,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 3000,
    "weight_decay": 2e-4,
}


In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

train_x = np.load("../data/Language/train_x.npy")
train_y = np.load("../data/Language/train_y.npy")
valid_x = np.load("../data/Language/valid_x.npy")
valid_y = np.load("../data/Language/valid_y.npy")
test_x = np.load('../data/Language/test_x.npy')
test_y = np.load('../data/Language/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 24 * 24)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 24 * 24)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 24 * 24)
test_tensor_y = torch.tensor(test_y).long()


train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)


train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
import wandb

wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-Language",
    name=f"trash",
    config=hyperparams
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: vanyamironov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, val_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [10]:
accuracy, params

(0.8337, 694139)

In [ ]:
torch.save(sparse_model, 'language.pt')

In [11]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 97.00, 85.71, 84.54, 83.40, 84.63, 84.57, 93.97, 90.12, 88.55, 87.65, 90.12, 76.83, 10],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))

styled_table

Model,Accuracy (%),Parameters
Ours,83.37,"694,139.0"
ResNet-18,97.00,"11,689,512.0"
AlexNet,85.71,"61,100,840.0"
VGG16,84.54,"138,357,544.0"
ConvNext,83.40,"88,591,464.0"
MNASNet,84.63,"4,383,312.0"
DenseNet,84.57,"28,681,000.0"
ResNeXt,93.97,"25,028,904.0"
PC-DARTS,90.12,nan
DrNAS,88.55,nan
